<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex04-perceptron-to-mlp/Ex04_06_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_04 · Notebook 06 — The Report

**Deep Learning for Engineering · Aalborg University · Part 1**

This notebook assembles the deliverable for Ex_04: a short markdown report in
which you choose a model and defend it.

## What is being marked

Two questions carry most of the marks, one from each lecture in the block.

> **L4.1 slide 22.** At equal parameter count, which did better — deeper or
> wider? And why do you think so?

> **L4.2 slide 22.** Which model would you deploy, and what would have to be true
> for that to be right?

Neither has a fixed correct answer. Both have a correct *shape* of answer: a
claim, the number that supports it, the spread that qualifies it, and the
condition under which it would stop being true. An answer with a number and no
qualification scores less than one with both, even when the number is the same.

Then the **four questions from L3.2 slide 2**, asked about the model you chose:

1. What is the input, precisely?
2. What is the loss — what single number was minimised?
3. Where did the data come from, and who paid for it?
4. What happens when it is wrong?

These four are how every exercise report in this course is marked, from Ex_03 to
Ex_12.

## How to use this notebook

Fill in the `ANSWERS` dictionary. Run the rest. It writes
`Ex04_outputs/Ex04_report.md` and prints it for checking.

---

## 0 · Your results, collected

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_4_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex04-perceptron-to-mlp/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import textwrap
from datetime import date

import numpy as np

import Ex_4_core as core

def load(name):
    path = os.path.join(core.OUTPUT_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} is missing — run the notebook that writes it first")
    return np.load(path, allow_pickle=True)

kinks = load("kinks.npz")
depth = load("depth_vs_width.npz")
reg = load("regularisation.npz")

kink_table = core.error_table(
    [[int(D), int(p), int(k), f"{m:.5f}"]
     for D, p, k, m in zip(kinks["D"], kinks["params"], kinks["kinks"],
                           kinks["mse"])],
    ["D", "parameters", "kinks in [0,1]", "training MSE"])

depth_table = core.error_table(
    [[str(n), int(p), f"{tr.mean():.5f} ± {tr.std():.5f}",
      f"{va.mean():.5f} ± {va.std():.5f}"]
     for n, p, tr, va in zip(depth["names"], depth["params"],
                             depth["final_train"], depth["best_val"])],
    ["configuration", "parameters", "final training MSE",
     "best validation MSE"])

reg_table = core.error_table(
    [[f"{l:g}", f"{t:.6f}", f"{v:.5f}", f"{te:.5f}", f"{tv:.2f}"]
     for l, t, v, te, tv in zip(reg["lambdas"], reg["train"], reg["val"],
                                reg["truth_err"], reg["tv"])],
    ["weight decay", "training MSE", "validation MSE", "error vs truth",
     "total variation"])

print(kink_table, "\n")
print(depth_table, "\n")
print(reg_table)
print(f"\nbest weight decay: {float(reg['best_lambda']):g}")
print(f"noise floor: {float(reg['noise_floor']):.5f}")

---

## 0b · Your personal seed

Every notebook in this exercise set fixes the seed to 0, so the printed
"what you should see" blocks are true on any machine. That is right for
checking your work and wrong for reporting it — with one seed the whole cohort
produces identical numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints in your answers where the questions ask for it. Your
supervisor can regenerate exactly these numbers from your study number alone,
so they are worth getting right and pointless to invent.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = core.personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own measurement campaign on the same underlying function.
x_you, y_you = core.wiggly_dataset(n=120, noise=0.06, seed=SEED)
NOISE_YOU = float(np.sqrt(np.mean((y_you - core.wiggly_truth(x_you)) ** 2)))
MEAN_YOU = float(y_you.mean())

xv_you, yv_you = core.lecture_validation(n=40, seed=SEED)
VAL_MEAN_YOU = float(yv_you.mean())

print()
print(f"  your realised noise level : {NOISE_YOU:.5f}   (drawn from 0.06)")
print(f"  your sample mean          : {MEAN_YOU:.5f}")
print(f"  your validation mean      : {VAL_MEAN_YOU:.5f}")

**What you should see.** Three tables — the kink sweep from notebook 03, the
depth-against-width comparison from notebook 04, and the regularisation sweep
from notebook 05 — plus your chosen weight decay.

If a `FileNotFoundError` appears, run the notebook named in the message to the
end. Each of 03, 04 and 05 writes its results as its last cell.

---

## 1 · What a good answer looks like

One worked example, so the expected level is not a guess. This is an answer to
the depth-against-width question, written at the standard the marking expects:

> At a budget of 1000 parameters, the two-hidden-layer network reached a best
> validation MSE of 0.0043 ± 0.0004 across five seeds against 0.0051 ± 0.0009 for
> the single wide layer, so the deeper shape was better by about fifteen per
> cent — but the seed-to-seed spread is a quarter of that difference, so I would
> report it as a weak effect rather than a clear one. The mechanism I would
> propose is the region-counting argument from L4.2: at fixed budget, depth buys
> more linear regions than width, because another layer costs $W^2$ where another
> $W$ units in one layer costs $3W$. I do not think this experiment demonstrates
> that mechanism, though, because all three shapes reached within a factor of two
> of the noise floor — the target was easy enough that none of them ran out of
> capacity, so what I measured was mostly optimisation behaviour. A budget of
> sixty parameters would test the claim properly.

Notice what that answer does: it gives the numbers, states the size of the
effect against the size of the noise, offers a mechanism, and then says why the
experiment does not actually establish the mechanism. The last clause is the one
that separates a good report from an average one.

---

## 2 · Your answers

Fill in every string. Replace the placeholder text entirely.

In [ ]:
# TODO: write your report. Every string below must be replaced.

PLACEHOLDER = "TODO: write this."

ANSWERS = {
    # --- the XOR story, briefly -------------------------------------------
    "xor": PLACEHOLDER,          # what the hidden layer bought, in your own words
    "xor_cost": PLACEHOLDER,     # what it cost — training, seeds, interpretability

    # --- L4.1 slide 22: the question that carries the marks ---------------
    "depth_vs_width": PLACEHOLDER,   # which did better, with numbers and spread
    "depth_mechanism": PLACEHOLDER,  # why you think so, and what the experiment
                                     # does and does not establish

    # --- L4.2 slide 22: the other question that carries the marks ---------
    "deploy": PLACEHOLDER,       # which model from notebook 05 you would deploy
    "conditions": PLACEHOLDER,   # what would have to be true for that to be right
    "abandon": PLACEHOLDER,      # what would make you change your mind

    # --- the four questions, about the model you chose to deploy ----------
    "input":   PLACEHOLDER,
    "loss":    PLACEHOLDER,
    "data":    PLACEHOLDER,
    "failure": PLACEHOLDER,

    # --- one sentence each ------------------------------------------------
    "kinks":     PLACEHOLDER,    # what the kink experiment changed in how you
                                 # picture a network
    "surprise":  PLACEHOLDER,    # what surprised you across the five notebooks
}

STUDENT_NAME = "TODO: your name"

if STUDENT_NAME.startswith("TODO") or any(t.startswith("TODO") for t in ANSWERS.values()):
    raise NotImplementedError(
        "Replace every placeholder above with your own prose, and put your name in "
        "STUDENT_NAME. The dictionary is defined, so you can edit it and re-run this "
        "cell as often as you like."
    )

## 3 · Check, assemble, save

In [ ]:
def check_answers(answers, name):
    problems = []
    if name.startswith("TODO"):
        problems.append("STUDENT_NAME")
    for key, text in answers.items():
        if text.startswith("TODO") or len(text.split()) < 8:
            problems.append(key)
    return problems

problems = check_answers(ANSWERS, STUDENT_NAME)
if problems:
    print("still to write (or shorter than eight words):")
    for p in problems:
        print("  -", p)
else:
    print("all sections written")

**What you should see.** A list of what is left, or `all sections written`.

---

In [ ]:
def build_report(a, name):
    w = lambda text: textwrap.fill(text, 78)
    out = []
    out.append("# Ex_04 — Perceptron to MLP")
    out.append("")
    out.append(f"**{name}** · Deep Learning for Engineering · {date.today().isoformat()}")
    out.append("")

    out.append("## 1 · XOR, and what a hidden layer bought")
    out.append("")
    out.append(w(a["xor"]))
    out.append("")
    out.append("**What it cost:**")
    out.append("")
    out.append(w(a["xor_cost"]))
    out.append("")

    out.append("## 2 · Capacity: kinks against units")
    out.append("")
    out.append(kink_table)
    out.append("")
    out.append(w(a["kinks"]))
    out.append("")

    out.append("## 3 · Depth against width, at equal budget")
    out.append("")
    out.append(depth_table)
    out.append("")
    out.append("**Which did better?**")
    out.append("")
    out.append(w(a["depth_vs_width"]))
    out.append("")
    out.append("**Why do I think so, and what this experiment does not show:**")
    out.append("")
    out.append(w(a["depth_mechanism"]))
    out.append("")

    out.append("## 4 · Overfitting and regularisation")
    out.append("")
    out.append(reg_table)
    out.append("")
    out.append(f"Chosen weight decay: {float(reg['best_lambda']):g}. "
               f"Noise floor: {float(reg['noise_floor']):.5f}. "
               f"Validation MSE without regularisation: "
               f"{float(reg['val_overfit']):.5f}; with early stopping: "
               f"{float(reg['val_early']):.5f}.")
    out.append("")
    out.append("**Which model would I deploy?**")
    out.append("")
    out.append(w(a["deploy"]))
    out.append("")
    out.append("**What would have to be true for that to be right:**")
    out.append("")
    out.append(w(a["conditions"]))
    out.append("")
    out.append("**What would make me change my mind:**")
    out.append("")
    out.append(w(a["abandon"]))
    out.append("")

    out.append("## 5 · The four questions, about the model I chose")
    out.append("")
    for key, question in zip(["input", "loss", "data", "failure"],
                             core.four_questions()):
        out.append(f"**{question}**")
        out.append("")
        out.append(w(a[key]))
        out.append("")

    out.append("## 6 · Note")
    out.append("")
    out.append("**What surprised me:** " + w(a["surprise"]))
    out.append("")
    return "\n".join(out)

report = build_report(ANSWERS, STUDENT_NAME)
print(report[:1600])
print("...")

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
report_path = os.path.join(core.OUTPUT_DIR, "Ex04_report.md")
with open(report_path, "w", encoding="utf-8") as fh:
    fh.write(report)

print("written:", report_path)
print(len(report.split()), "words")

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex04_report.md into Ex04_report.pdf, with any figure
# saved as Ex04_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex04_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex04_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex04_report.pdf")
print("written Ex04_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex04_report.pdf")
except ImportError:
    pass


**What you should see.** A path ending in `Ex04_outputs/Ex04_report.md` and a
word count between about 700 and 1200 for a complete report.

Submit the markdown file together with two figures:

- the fifteen-panel loss grid from notebook 04, section 3,
- the three-fits figure from notebook 05, section 7.

Both are required. Every training run in this course is reported with its
training and validation curves on the same axes, and those two figures are the
evidence that you did.

---

## 4 · Where this goes next

You have now built, by hand and in PyTorch, the object the rest of the course
uses. Three things you did here recur immediately.

**The piecewise-linear picture** from notebook 03 is why Part 2 uses `tanh`: a
physics-informed loss contains second derivatives of the network, and the second
derivative of a ReLU network is zero almost everywhere. L4.1 slide 12 says so
eight weeks early, and this is the notebook that makes it obvious.

**Both curves on the same axes** is not an Ex_04 convention. Every training run
in L5, L6 and all of Part 2 is reported this way.

**"What would have to be true for this to be right"** is the question the marking
is built around, from here to Ex_12. Ex_03 asked it about a cooling curve, Ex_04
about a capacity choice, and Ex_12 will ask it about a power grid.

L5 is convolutional, graph, sequence and attention models — four architectures,
all of them the same object with a different rule about which weights are shared.
Nothing you learned here is replaced.